# 3W Workshop: Multiclass Sample Classification

Build a sample-level classifier that predicts the operational label **0 through 9** for each labeled observation.

**Audience:** Python users who know basic pandas and classification concepts.

**Learning goals:** load fixed instance splits with 3W Toolkit, apply the
workshop label rules, train a default model five times, and summarize the
validation results. No hyperparameter search is performed.

## Prerequisites and outline

Prerequisites:

- install the repository with `pip install -e .`;
- make the complete 3W Dataset v2.0.0 available under `dataset/`, or set
  `THREE_W_DATASET_PATH`;
- optionally set `THREE_W_WORKSHOP_MAX_INSTANCES_PER_CLASS=10` for a quick
  class- and split-complete smoke run.

Outline:

1. Resolve the repository, dataset, and fixed split lists.
2. Remove unlabeled rows and map transient labels to steady fault labels.
3. Use 3W Toolkit signal cleanup and normalization.
4. Fit a fresh default decision tree on each of the five fixed splits.
5. Review metrics and the aggregated confusion matrix.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

from ThreeWToolkit.core.enums import ModelTypeEnum
from ThreeWToolkit.dataset import ParquetDataset, ParquetDatasetConfig
from ThreeWToolkit.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from ThreeWToolkit.models.sklearn_models import (
    SklearnModels,
    SklearnModelsConfig,
)
from ThreeWToolkit.utils.data_utils import (
    GLOBAL_AVERAGES,
    default_data_cleanup,
    default_data_normalization,
    get_config_dataset_ini,
)

def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at root or by the notebook."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "toolkit" / "ThreeWToolkit"
        ).is_dir():
            return candidate
    raise FileNotFoundError("Could not find the 3W repository root.")


def load_env_file(path: Path) -> None:
    """Load simple KEY=VALUE entries without overriding shell variables."""
    if not path.is_file():
        return
    for line_number, raw_line in enumerate(
        path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line.removeprefix("export ").lstrip()
        key, separator, value = line.partition("=")
        key = key.strip()
        if not separator or not key.isidentifier():
            raise ValueError(f"Invalid .env entry at {path}:{line_number}")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {'\"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def configured_path(variable: str, default: Path) -> Path:
    """Resolve an optional environment path from the repository root."""
    value = os.getenv(variable)
    path = Path(value).expanduser() if value else default
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


REPO_ROOT = find_repo_root()
ENV_FILE = REPO_ROOT / ".env"
load_env_file(ENV_FILE)
WORKSHOP_DIR = REPO_ROOT / "toolkit" / "demos" / "workshop"
SPLIT_DIR = configured_path(
    "THREE_W_WORKSHOP_SPLIT_DIR", WORKSHOP_DIR / "splits"
)
DATASET_PATH = configured_path("THREE_W_DATASET_PATH", REPO_ROOT / "dataset")
RANDOM_SEED = int(os.getenv("THREE_W_WORKSHOP_RANDOM_SEED", "42"))
SAMPLES_PER_LABEL_PER_INSTANCE = int(
    os.getenv("THREE_W_WORKSHOP_SAMPLES_PER_LABEL_PER_INSTANCE", "100")
)
MAX_INSTANCES_PER_CLASS = int(
    os.getenv("THREE_W_WORKSHOP_MAX_INSTANCES_PER_CLASS", "0")
)

print(f"Configuration: {ENV_FILE if ENV_FILE.is_file() else 'built-in defaults'}")
print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_PATH}")
print(
    "Instance limit per class: "
    + (str(MAX_INSTANCES_PER_CLASS) if MAX_INSTANCES_PER_CLASS else "all")
)

## 1. Load the fixed instance splits

Each text file contains the validation instances for one run. The other four
lists form training. Because assignment is at the instance level, samples from
one Parquet file never appear in both training and validation.

In [ ]:
def read_validation_lists(split_dir: Path) -> dict[int, set[str]]:
    """Read the five one-path-per-line toolkit file lists."""
    validation_lists = {}
    for split_number in range(1, 6):
        path = split_dir / f"validation_split_{split_number}.txt"
        instances = {
            line.strip()
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        }
        validation_lists[split_number] = instances

    all_entries = [
        instance for instances in validation_lists.values() for instance in instances
    ]
    if len(all_entries) != len(set(all_entries)):
        raise ValueError("An instance occurs in more than one validation list.")
    return validation_lists


def select_smoke_subset(
    validation_lists: dict[int, set[str]], max_per_class: int
) -> list[str]:
    """Optionally keep a class- and split-complete subset for a quick run."""
    all_instances = sorted(set().union(*validation_lists.values()))
    if max_per_class <= 0:
        return all_instances
    if max_per_class < 5:
        raise ValueError("The smoke limit must be 0 or at least 5 per class.")

    quotient, remainder = divmod(max_per_class, 5)
    selected = []
    for event_class in range(10):
        prefix = f"{event_class}/"
        for split_number, instances in validation_lists.items():
            count = quotient + int(split_number <= remainder)
            candidates = sorted(
                instance for instance in instances if instance.startswith(prefix)
            )
            if len(candidates) < count:
                raise ValueError(
                    f"Class {event_class}, split {split_number} has only "
                    f"{len(candidates)} instances; requested {count}."
                )
            selected.extend(candidates[:count])
    return sorted(selected)


validation_lists = read_validation_lists(SPLIT_DIR)
instance_files = select_smoke_subset(validation_lists, MAX_INSTANCES_PER_CLASS)

print(f"Using {len(instance_files):,} instance files.")
pd.Series(
    {split: len(files) for split, files in validation_lists.items()},
    name="all_validation_instances",
).to_frame()

## 2. Apply the workshop cleaning and label rules

The loader intentionally configures `ParquetDataset(clean_data=False)` so raw
missing labels are still visible. It then:

1. drops missing labels;
2. maps transient labels with the toolkit's configured offset (`105 → 5`);
3. applies the toolkit's canonical signal cleanup and global normalization;
4. keeps up to 100 evenly spaced samples per observed label per instance so the
   workshop remains practical without changing model hyperparameters.

In [ ]:
FEATURE_COLUMNS = list(GLOBAL_AVERAGES)
TRANSIENT_OFFSET = int(get_config_dataset_ini()["TRANSIENT_OFFSET"])


def evenly_spaced_positions(positions: np.ndarray, limit: int) -> np.ndarray:
    """Keep temporal coverage while bounding workshop memory and runtime."""
    if len(positions) <= limit:
        return positions
    selected = np.linspace(0, len(positions) - 1, limit, dtype=int)
    return positions[selected]


def load_workshop_samples(
    dataset_path: Path,
    files: list[str],
    *,
    binary: bool,
    samples_per_label_per_instance: int,
) -> tuple[pd.DataFrame, int]:
    """Load, clean, normalize, relabel, and sample 3W observations."""
    dataset = ParquetDataset(
        ParquetDatasetConfig(
            path=dataset_path,
            split="list",
            file_list=files,
            columns=FEATURE_COLUMNS,
            target_column="class",
            clean_data=False,
            seed=RANDOM_SEED,
        )
    )

    tables = []
    unlabeled_removed = 0
    for item in tqdm(dataset.iterfiles(), total=len(dataset), unit="instance"):
        signal = item["signal"]
        raw_label = item["label"]["class"]
        valid_positions = np.flatnonzero(raw_label.notna().to_numpy())
        unlabeled_removed += len(raw_label) - len(valid_positions)
        if not len(valid_positions):
            continue

        signal = signal.iloc[valid_positions].copy()
        labels = (
            raw_label.iloc[valid_positions].astype("int64").to_numpy()
            % TRANSIENT_OFFSET
        )

        allowed = np.isin(labels, np.arange(10))
        signal = signal.iloc[np.flatnonzero(allowed)].copy()
        labels = labels[allowed]

        # Use the toolkit's canonical signal cleaning and global normalization.
        signal = default_data_cleanup(signal)
        signal = default_data_normalization(signal).fillna(0.0)

        kept_positions = np.concatenate(
            [
                evenly_spaced_positions(
                    np.flatnonzero(labels == label),
                    samples_per_label_per_instance,
                )
                for label in sorted(np.unique(labels))
            ]
        )
        kept_positions.sort()
        sample = signal.iloc[kept_positions].copy()
        target = labels[kept_positions]
        if binary:
            target = (target != 0).astype("int8")
        sample["target"] = target
        sample["instance"] = item["file_name"].as_posix()
        tables.append(sample)

    if not tables:
        raise RuntimeError("No labeled samples were loaded.")
    return pd.concat(tables, ignore_index=True), unlabeled_removed

In [ ]:
samples, unlabeled_removed = load_workshop_samples(
    DATASET_PATH,
    instance_files,
    binary=False,
    samples_per_label_per_instance=SAMPLES_PER_LABEL_PER_INSTANCE,
)
print(f"Removed {unlabeled_removed:,} unlabeled observations before cleaning.")
samples.groupby("target").size().rename("samples").to_frame()

## 3. Fit and evaluate all five splits

The model is the 3W Toolkit wrapper around scikit-learn's
`DecisionTreeClassifier`. Its model hyperparameters stay at their defaults; the
toolkit only fixes `random_state=42` for reproducibility. Macro-F1 and balanced accuracy give every class equal weight, making them more informative than raw accuracy for the imbalanced 3W labels.

In [ ]:
def fresh_default_model() -> SklearnModels:
    """Create a fresh toolkit decision tree with default hyperparameters."""
    return SklearnModels(
        SklearnModelsConfig(
            model_type=ModelTypeEnum.DECISION_TREE,
            random_seed=RANDOM_SEED,
        )
    )


def evaluate_five_splits(
    samples: pd.DataFrame,
    split_files: dict[int, set[str]],
    labels: list[int],
) -> tuple[pd.DataFrame, np.ndarray, dict]:
    """Fit and evaluate one fresh default model on every fixed split."""
    feature_columns = [
        column for column in samples.columns if column not in {"target", "instance"}
    ]
    available_instances = set(samples["instance"])
    rows = []
    aggregate_confusion = np.zeros((len(labels), len(labels)), dtype=int)
    effective_params = None

    for split_number in range(1, 6):
        validation_instances = split_files[split_number] & available_instances
        validation_mask = samples["instance"].isin(validation_instances)
        train_mask = ~validation_mask

        x_train = samples.loc[train_mask, feature_columns]
        y_train = samples.loc[train_mask, "target"]
        x_validation = samples.loc[validation_mask, feature_columns]
        y_validation = samples.loc[validation_mask, "target"]

        model = fresh_default_model()
        effective_params = model.get_params()
        model.fit(x_train, y_train)
        prediction = model.predict(x_validation)

        row = {
            "split": split_number,
            "train_samples": int(train_mask.sum()),
            "validation_samples": int(validation_mask.sum()),
            "accuracy": accuracy_score(y_validation, prediction),
            "balanced_accuracy": balanced_accuracy_score(y_validation, prediction),
            "macro_f1": f1_score(
                y_validation,
                prediction,
                labels=labels,
                average="macro",
                zero_division=0,
            ),
        }
        rows.append(row)
        aggregate_confusion += confusion_matrix(y_validation, prediction, labels=labels)

    return pd.DataFrame(rows), aggregate_confusion, effective_params

In [ ]:
fold_results, aggregate_confusion, effective_params = evaluate_five_splits(
    samples, validation_lists, labels=list(range(10))
)
display(fold_results.round(4))
display(fold_results.mean(numeric_only=True).rename("five_split_mean").to_frame())

print("DecisionTree parameters used in every split (no search):")
effective_params

## 4. Aggregate the confusion matrices

Summing counts before row normalization gives every validation observation one
vote across the five non-overlapping validation folds.

In [ ]:
row_totals = aggregate_confusion.sum(axis=1, keepdims=True)
normalized_confusion = np.divide(
    aggregate_confusion,
    row_totals,
    out=np.zeros_like(aggregate_confusion, dtype=float),
    where=row_totals != 0,
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=list(range(10)),
    yticklabels=list(range(10)),
    ax=ax,
)
ax.set(
    xlabel="Predicted class", ylabel="True class", title="Five-split confusion matrix"
)
plt.show()

## Exercise: find the hardest classes

Recall is the diagonal of the row-normalized confusion matrix. Rank the classes
to identify which event labels the default model recognizes least reliably.

In [ ]:
# Answer scaffold
recall_by_class = pd.Series(
    np.diag(normalized_confusion),
    index=list(range(10)),
    name="recall",
)
recall_by_class.sort_values().to_frame()

## Pitfalls and extensions

- **Do not fill missing labels before filtering.** The workshop rule is to
  remove them; the toolkit's default label processing uses forward/backward
  filling, so this tutorial handles labels explicitly.
- **Do not split rows randomly.** Neighboring samples are highly correlated.
  These folds keep whole instances together and also group real files from the
  same well and day.
- This default tree is a teaching baseline, not a production model.

Optional extension: keep the split files fixed and compare a window-based
feature extractor or a stricter whole-well validation design. Treat that as a
new experiment rather than tuning on these validation results.